<a href="https://colab.research.google.com/github/alexandrumoldovan1/housing-prices-ml/blob/main/notebooks/05b_evaluation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install ML libraries
!pip install shap xgboost lightgbm catboost -q

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# ML libraries
import xgboost
import lightgbm
import catboost
import shap

# Sklearn imports
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00
Libraries imported successfully!


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths
DRIVE_PATH = '/content/drive/MyDrive/ColabProjects/housing-prices-ml'
PROCESSED_DATA_PATH = f'{DRIVE_PATH}/processed_data'
MODELS_PATH = f'{DRIVE_PATH}/models'
OUTPUTS_PATH = f'{DRIVE_PATH}/outputs'

# Load capped data (the final dataset used for best model)
X_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train_capped.parquet')
X_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test_capped.parquet')
y_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_train_capped.parquet').squeeze()
y_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_test_capped.parquet').squeeze()

print(f"Data loaded:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")
print(f"\nPrice range (test, capped at $10M):")
print(f"  Min:    ${np.expm1(y_test).min():,.0f}")
print(f"  Max:    ${np.expm1(y_test).max():,.0f}")
print(f"  Median: ${np.expm1(y_test).median():,.0f}")

Mounted at /content/drive
Data loaded:
  X_train: (97683, 43)
  X_test:  (51679, 43)
  y_train: (97683,)
  y_test:  (51679,)

Price range (test, capped at $10M):
  Min:    $17,267
  Max:    $10,000,000
  Median: $860,000


In [3]:
# Load all capped models
models = {
    'XGBoost': joblib.load(f'{MODELS_PATH}/xgboost_capped.pkl'),
    'LightGBM': joblib.load(f'{MODELS_PATH}/lightgbm_capped.pkl'),
    'CatBoost': joblib.load(f'{MODELS_PATH}/catboost_capped.pkl'),
    'Stacking': joblib.load(f'{MODELS_PATH}/stacking_capped.pkl'),
}

print(f"Loaded {len(models)} models:")
for name, model in models.items():
    print(f"  - {name}: {type(model).__name__}")

Loaded 4 models:
  - XGBoost: XGBRegressor
  - LightGBM: LGBMRegressor
  - CatBoost: CatBoostRegressor
  - Stacking: StackingRegressor


In [4]:
# Evaluate all capped models on test set
results = []

for name, model in models.items():
    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    r2 = r2_score(y_test, y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results.append({
        'Model': name,
        'R2_test': r2,
        'RMSE': rmse,
        'MAE': mae
    })
    print(f"{name:15s}: R²={r2:.4f}, RMSE=${rmse:,.0f}, MAE=${mae:,.0f}")

results_capped = pd.DataFrame(results)
print(f"\nResults shape: {results_capped.shape}")

XGBoost        : R²=0.5623, RMSE=$933,006, MAE=$457,569
LightGBM       : R²=0.5539, RMSE=$937,430, MAE=$463,637
CatBoost       : R²=0.5514, RMSE=$949,645, MAE=$468,454
Stacking       : R²=0.5673, RMSE=$934,968, MAE=$456,300

Results shape: (4, 4)
